# ME 498/598 Generative Models Demo

<div style="font-size: small;">
License Terms:
These Python demos are licensed under a <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">Creative Commons Attribution-NonCommercial-NoDerivatives 4.0 International License</a>. They are intended for educational use only in ME 498/598: Applied AI for Engineering and Design at the University of Illinois Urbana-Champaign. You may not share or distribute them publicly, use them for commercial purposes, or provide them to industry or other entities without permission from the instructor (regnwttr@illinois.edu).
</div>

In this demo, we design structural brackets, first with a classical optimizer and then with a generative model that has learned from thousands of previous designs. This is also the design problem of CP3. We cover:

- **Part A: Topology optimization with SIMP**, the classical way to design one bracket for one set of requirements
- **Part B: A conditional diffusion model (DDPM)** that generates a bracket for any set of requirements in a fraction of a second
- **Part C: Guidance**, steering the diffusion model toward stiffer brackets with compliance gradients from FEA

## **Setting Things Up**


In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import trange

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## **The Design Problem: Plate Brackets**

Our brackets are flat parts cut from an **80 mm × 80 mm** metal plate. Each bracket is bolted down at **three clamped supports** on its boundary and carries a **load** applied through a pin hole (the *load disc*). We want the bracket to be as **stiff** as possible while using at most a given fraction of the plate's area.

- **Design:** an 80 × 80 binary image, where 1 is material and 0 is empty space. One pixel is 1 mm.
- **Condition** (the requirements): 8 numbers.
  - `load_x, load_y`: the centre of the load disc (radius 3 mm), in mm, with x to the right and y *down* (like the image's rows and columns).
  - `force_x, force_y`: the direction of the load, a unit vector in the same frame.
  - `vf_cap`: the volume fraction cap, the largest allowed share of the plate covered by material (between 0.2 and 0.5).
  - `sup_1, sup_2, sup_3`: the centre of each 8 mm clamped support, as a distance along the plate's boundary, clockwise from the top-left corner (top edge 0–80, right 80–160, bottom 160–240, left 240–320).
- **Performance:** the **compliance**, the work done by the load, $C = F^\top u$, where $u$ is the displacement computed by finite element analysis (FEA). Lower compliance means a stiffer bracket. This is the topology-optimization meaning of compliance, an energy that grows with the square of the load, not the 1/stiffness of a spring (units of m/N); with our unit load, the two give the same number. The load is shared equally by the nodes of the load disc that touch material. Empty space is modelled as a material a million times softer than the plate, so the FEA always works. If the material doesn't connect the load to the supports, the load has to go through this "air", and the compliance becomes enormous: values above roughly $10^4$ mean the load path is broken.

All of this is implemented in `cp3_public.py`, which you'll also use in CP3.

In [ ]:
import cp3_public as P                                           # the CP3 utilities, in the repository root

X, cond, C = P.load_designs('datasets/cp3_brackets.npz')        # [n x 80 x 80] designs of 0/1, [n x 8], [n]

# Hold out 10% of the designs as a test set
X_train, X_test, cond_train, cond_test, C_train, C_test = train_test_split(X, cond, C, test_size=0.1, random_state=0)
print(f'{len(X)} designs: {len(X_train)} for training, {len(X_test)} for testing')
print('Condition of the first training design:')
for name, value in zip(P.COND_COLS, cond_train[0]):
    print(f'  {name:8s} {value:7.2f}')

plt.figure(figsize=(3.5, 3.5))
P.plot_design(X_train[0], cond_train[0], title=f'First training design: C = {C_train[0]:.3g}')
plt.show()

Supports are drawn in blue; the load disc and the load direction in red. Let's look at some more designs:

In [ ]:
P.plot_designs(X_train[:18], cond_train[:18], [f'C = {C_train[i]:.3g}, vf_cap = {cond_train[i, 4]:.2f}' for i in range(18)])
plt.show()

Some of these are strong designs; others aren't, and a few don't even touch the load disc. Let's look at the compliance and how much of its volume budget each design uses:

In [ ]:
vf_ratio = X_train.mean(axis=(1, 2)) / cond_train[:, 4]     # share of the plate covered, relative to the cap

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
sns.kdeplot(x=np.log10(C_train), fill=True, ax=axs[0])
axs[0].axvline(4, color='k', ls='--', lw=1, label='Approx. Threshold for Degenerate Designs')
axs[0].set_xlabel('log10(compliance)')
axs[0].legend()
sns.kdeplot(x=vf_ratio, fill=True, ax=axs[1])
axs[1].axvline(1, color='k', ls='--', lw=1, label='volume cap')
axs[1].set_xlabel('volume fraction used / vf_cap'); axs[1].legend()
plt.tight_layout(); plt.show()

Compliance spans several orders of magnitude, so we will mostly work with its logarithm. The bump on the far right is designs whose load path is broken. Most designs use close to their full volume budget, and a fair number go slightly over it.

## **Part A: Topology Optimization with SIMP**

The classical way to design a bracket is **topology optimization**. The most popular method, **SIMP** (Solid Isotropic Material with Penalization), works like this:

1. Give each of the 6,400 elements a *density* between 0 and 1, starting from a uniform gray plate at the volume cap.
2. Run FEA, with each element's stiffness scaled by $\text{density}^3$. The exponent makes gray, half-dense material inefficient, which pushes the densities toward 0 or 1.
3. Compute how the compliance changes with every density. This sensitivity is cheap: it comes straight out of the FEA solution.
4. Move material from where it helps least to where it helps most, keeping the total volume at the cap, and repeat.

A *density filter* smooths the densities so that the design doesn't break up into a checkerboard. `P.simp` implements all this in about 30 lines; have a look if you're curious. Let's run it on one test condition:

In [ ]:
k = 0                                   # index of a test condition
cond = cond_test[k]

t0 = time.time()
x_simp = P.simp(cond)                   # [80 x 80] densities between 0 and 1
t_simp = time.time() - t0
C_simp = P.compliance(x_simp, cond)     # compliance() thresholds the densities at 0.5
print(f'SIMP took {t_simp:.1f} s')

fig, axs = plt.subplots(1, 3, figsize=(10, 3.6))
axs[0].imshow(x_simp, cmap='gray_r', vmin=0, vmax=1); axs[0].axis('off'); axs[0].set_title('SIMP densities', fontsize=8)
P.plot_design(x_simp, cond, axs[1], title=f'SIMP, thresholded: C = {C_simp:.3g}')
P.plot_design(X_test[k], cond, axs[2], title=f'Dataset design: C = {C_test[k]:.3g}')
plt.tight_layout(); plt.show()

SIMP produces a clean, efficient structure, but it takes tens of seconds of FEA for every single condition, and it starts over from scratch each time. A **conditional generative model** instead learns the mapping from condition to design once, from data, and then produces a new design in a fraction of a second.

### Representing the Condition

We could feed the 8 condition numbers straight into a network. But CNNs reason about *where* things are on the grid, so it helps to draw the condition on the same 80 × 80 grid as the design. `P.condition_image` draws it as 6 channels:

- **support**: 1 on the elements touching a clamped support
- **force_x**, **force_y**: the load direction, on the elements of the load disc
- **vf_cap**: the volume cap, the same value everywhere
- **dist_load**, **dist_support**: the distance from every element to the load and to the nearest support

This also takes care of two awkward properties of the 8 numbers: the three supports are an unordered set (listing them in a different order is the same problem), and positions 0 and 320 on the boundary are the same point.

Why the two distance channels? A convolution only looks at a small neighbourhood, so a pixel near a support can't directly see a load disc 60 pixels away. The distance fields hand that global information to every pixel. Without them, our diffusion model below reaches the load in only about half of its designs.

In [ ]:
img = P.condition_image(cond)            # [6 x 80 x 80]

fig, axs = plt.subplots(1, 7, figsize=(16, 2.8))
P.plot_design(X_test[k], cond, axs[0], title='Design')
for i in range(6):
    axs[i + 1].imshow(img[i], cmap='RdBu_r', vmin=-1, vmax=1)
    axs[i + 1].set_title(P.COND_CHANNELS[i], fontsize=9); axs[i + 1].axis('off')
plt.tight_layout(); plt.show()

We convert all designs and conditions to tensors on the GPU. Designs are scaled to $[-1, 1]$, the usual range for diffusion models.

In [ ]:
def to_tensors(X, conds):
    x = torch.tensor(X).unsqueeze(1).to(device) * 2 - 1                                     # [n x 1 x 80 x 80], in [-1, 1]
    c = torch.tensor(np.stack([P.condition_image(cd) for cd in conds])).to(device)          # [n x 6 x 80 x 80]
    return x, c

x_train, c_train = to_tensors(X_train, cond_train)
x_test, c_test = to_tensors(X_test, cond_test)

## **Part B: A Conditional Diffusion Model (DDPM)**

Recall from lecture: a **denoising diffusion probabilistic model** (DDPM) learns to undo noise.

- **Forward process (fixed):** take a clean design $x_0$ and mix it with Gaussian noise $\epsilon$: $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \epsilon$. At step $t = 0$ it's almost clean; by $t = T$ it's pure noise.
- **Training:** pick a random design, step $t$ and noise $\epsilon$, and train a network to undo the noise.
- **Generating:** start from pure noise and repeatedly remove a bit of it.

In lecture, the network guessed the noise, $\epsilon_\theta(x_t, t)$. Here it guesses the **clean design** directly, $\hat x_0 = f_\theta(x_t, t)$, with the plain regression loss $\|x_0 - f_\theta(x_t, t)\|^2$. The two are equivalent on paper: given $x_t$, either guess determines the other through $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \epsilon$. In practice, guessing $x_0$ learns our crisp black-and-white designs much faster: after the same training time, the noise-guessing version reached the load in 90% of designs with twice the compliance, the clean-design version in 98%. It also hands us $\hat x_0$ directly, which we'll steer in Part C.

To make the model **conditional**, we simply give the network the condition channels next to the noisy design: its input is 1 + 6 = 7 channels. The network is a small **U-Net**: it downsamples the image to capture the overall layout, upsamples it back, and passes *skip connections* across so that fine detail isn't lost. Every block is also told the step $t$, so that it knows how noisy its input is.

One trick keeps it fast: the U-Net works on **2 × 2 pixel blocks** (`pixel_unshuffle`). The 80 × 80 × 7 input becomes 40 × 40 × 28, and the output is unfolded back at the end (`pixel_shuffle`). Almost all of a U-Net's compute is at its highest resolution, so this makes training about 4× faster, and the designs don't need single-pixel detail anyway.

In [ ]:
def timestep_embedding(t, dim=32):
    """Sinusoidal features of the diffusion step, like the positional encodings of a transformer."""
    freqs = torch.exp(-np.log(10000) * torch.arange(dim // 2, device=t.device) / (dim // 2))
    args = t.float()[:, None] * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class ResBlock(nn.Module):
    def __init__(self, c_in, c_out, t_dim):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(c_out, c_out, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(8, c_out)
        self.norm2 = nn.GroupNorm(8, c_out)
        self.t_proj = nn.Linear(t_dim, c_out)
        self.skip = nn.Conv2d(c_in, c_out, kernel_size=1) if c_in != c_out else nn.Identity()

    def forward(self, x, t_emb):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.t_proj(t_emb)[:, :, None, None]       # tell every pixel how noisy the input is
        h = F.silu(self.norm2(self.conv2(h)))
        return h + self.skip(x)                            # residual connection


class UNet(nn.Module):
    def __init__(self, c_in=7, c_out=1, widths=(16, 32, 64, 64), t_dim=128):
        super(UNet, self).__init__()
        self.t_mlp = nn.Sequential(nn.Linear(32, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        self.down = nn.ModuleList([ResBlock(4 * c_in if i == 0 else widths[i - 1], w, t_dim) for i, w in enumerate(widths)])
        self.up = nn.ModuleList([ResBlock(widths[i + 1] + widths[i], widths[i], t_dim) for i in reversed(range(len(widths) - 1))])
        self.out = nn.Conv2d(widths[0], 4 * c_out, kernel_size=1)

    def forward(self, x, t):
        t_emb = self.t_mlp(timestep_embedding(t))
        x = F.pixel_unshuffle(x, 2)                        # 2 x 2 pixel blocks: [7 x 80 x 80] -> [28 x 40 x 40]
        skips = []
        for i, block in enumerate(self.down):
            if i > 0:
                x = F.avg_pool2d(x, 2)                     # 40 -> 20 -> 10
            x = block(x, t_emb)
            skips.append(x)
        for block, skip in zip(self.up, reversed(skips[:-1])):
            x = F.interpolate(x, scale_factor=2)           # 10 -> 20 -> 40
            x = block(torch.cat([x, skip], dim=1), t_emb)  # skip connection: reuse the detail from the way down
        return F.pixel_shuffle(self.out(x), 2)             # back to [1 x 80 x 80]

Here is the layout of our U-Net, with the shape of the data after every block. Each down block halves the resolution and adds channels; each up block doubles the resolution and gets the matching down block's output passed across (the skip connection):

In [ ]:
P.print_unet(UNet())

We use the standard image schedule: $T = 1000$ steps with $\beta_t$ rising linearly from $10^{-4}$ to $0.02$.

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)
alpha_bar = torch.cumprod(1 - betas, dim=0)

fig, axs = plt.subplots(1, 6, figsize=(13, 2.4))
x0 = x_train[:1]
for ax, t in zip(axs, [0, 50, 150, 300, 600, 999]):
    xt = alpha_bar[t].sqrt() * x0 + (1 - alpha_bar[t]).sqrt() * torch.randn_like(x0)
    ax.imshow(xt[0, 0].cpu(), cmap='gray_r', vmin=-1, vmax=1); ax.set_title(f't = {t}', fontsize=9); ax.axis('off')
plt.suptitle('The forward process on one design', fontsize=10); plt.tight_layout(); plt.show()

Two training details:

- **Mixed precision** (`torch.autocast`): the GPU computes in 16-bit floats where it's safe, which roughly doubles the speed.
- **Cosine learning-rate schedule:** the learning rate decays smoothly to zero over training. Diffusion models are sensitive to the noise in the last few updates, and small final steps give noticeably cleaner samples. (A popular alternative is to generate with an *exponential moving average* of the weights. Either one works here, but without both, samples get worse.)

In [ ]:
def train_ddpm(model, x, c, num_epochs=60, batch_size=128, learning_rate=1e-3):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    steps_per_epoch = len(x) // batch_size
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_epochs * steps_per_epoch)
    scaler = torch.amp.GradScaler(enabled=device.type == 'cuda')
    losses = []
    for epoch in trange(num_epochs):
        model.train()
        order = torch.randperm(len(x), device=device)
        epoch_loss = 0.0
        for i in range(steps_per_epoch):
            idx = order[i * batch_size:(i + 1) * batch_size]
            x0, cond = x[idx], c[idx]
            t = torch.randint(0, T, (len(idx),), device=device)          # a random step for every design
            eps = torch.randn_like(x0)
            ab = alpha_bar[t][:, None, None, None]
            xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps                   # forward process, in one jump
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                x0_guess = model(torch.cat([xt, cond], dim=1), t)
            loss = F.mse_loss(x0_guess.float(), x0)                       # guess the clean design
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            epoch_loss = epoch_loss + loss.detach()                        # summed on the GPU: .item() every step would stall it
        losses.append(epoch_loss.item() / steps_per_epoch)
    return model.eval(), losses

Training takes about 5 minutes on a Colab T4 GPU (about 2 on a recent desktop GPU).

In [ ]:
torch.manual_seed(0)
ddpm, losses = train_ddpm(UNet(), x_train, c_train, num_epochs=60)

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel('epoch'); plt.ylabel('mean loss'); plt.yscale('log'); plt.tight_layout(); plt.show()

**Generating designs.** DDPM's own sampler takes all 1,000 steps. Instead, we use **DDIM** (see lecture), which reuses the same trained network but jumps through the steps in big strides without adding fresh noise. 50 steps are plenty.

At each step, the network gives us its guess of the clean design $\hat x_0$. The matching noise guess is $\hat\epsilon = (x_t - \sqrt{\bar\alpha_t}\, \hat x_0) / \sqrt{1 - \bar\alpha_t}$, and DDIM moves to the next, less noisy step by mixing $\hat x_0$ and $\hat\epsilon$ at that step's noise level. We'll come back to $\hat x_0$ in Part C, where we nudge it toward stiffer designs (the `guide` argument).

In [ ]:
@torch.no_grad()
def sample(model, c, num_steps=50, guide=None, seed=0, history=None):
    """Generate one design per condition image in c with DDIM. Returns [n x 1 x 80 x 80] in [-1, 1].
    guide (Part C): a function that nudges the clean-design guess, applied during the first 70% of the steps.
    history: if a list, (x_t, clean-design guess) of every step is appended to it."""
    model.eval()
    generator = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(len(c), 1, 80, 80, device=device, generator=generator)   # start from pure noise
    steps = torch.linspace(T - 1, 0, num_steps, device=device).long()
    for i, t in enumerate(steps):
        ab = alpha_bar[t]
        ab_next = alpha_bar[steps[i + 1]] if i + 1 < num_steps else torch.tensor(1.0, device=device)
        x0 = model(torch.cat([x, c], dim=1), t.repeat(len(x))).clamp(-1, 1)    # current guess of the clean design
        if guide is not None and i < 0.7 * num_steps:
            x0 = guide(x0)                                                     # Part C: nudge the guess
        if history is not None:
            history.append((x.cpu(), x0.cpu()))
        eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()                          # the noise that goes with that guess
        x = ab_next.sqrt() * x0 + (1 - ab_next).sqrt() * eps                   # DDIM step to the next noise level
    return x


def to_designs(x):
    return (x[:, 0] > 0).cpu().numpy()        # threshold at 0: [n x 80 x 80] booleans

We evaluate on 100 test conditions the model has never seen. For each, we generate one design and compute its true compliance with FEA, then compare it with the dataset's design for the same condition.

In [ ]:
cond_eval, c_eval, C_data = cond_test[:100], c_test[:100], C_test[:100]

t0 = time.time()
D_ddpm = to_designs(sample(ddpm, c_eval))
t_ddpm = (time.time() - t0) / len(D_ddpm)
C_ddpm, vf_ddpm = P.evaluate(D_ddpm, cond_eval)        # FEA compliance, volume / cap

print(f'Generation time per design: {t_ddpm * 1000:.0f} ms (SIMP: {t_simp:.1f} s)')
print(f'Broken load path (compliance > 1e4): {(C_ddpm > 1e4).mean():.0%}')
print(f'Median compliance, generated / dataset design: {np.median(C_ddpm / C_data):.2f}')
print(f'Over the volume cap: {(vf_ddpm > 1).mean():.0%}')

In [ ]:
P.plot_designs(np.concatenate([X_test[:6], D_ddpm[:6]]), np.concatenate([cond_eval[:6]] * 2),
               [f'Dataset: C = {C_data[j]:.3g}' for j in range(6)] + [f'DDPM: C = {C_ddpm[j]:.3g}' for j in range(6)])
plt.show()

Let's watch DDIM at work on three conditions. For each, the top row shows the noisy sample $x_t$ and the bottom row the network's guess of the clean design $\hat x_0$ at that step. The overall layout is settled early; the later steps refine the edges.

In [ ]:
history = []
sample(ddpm, c_eval[:3], history=history)
show_steps = [0, 5, 10, 20, 30, 49]

images, conds, titles = [], [], []
for d in range(3):
    for k in range(2):                                     # k = 0: noisy sample x_t, k = 1: the network's guess
        images += [(history[s][k][d, 0] + 1) / 2 for s in show_steps]
        conds += [cond_eval[d]] * len(show_steps)
        titles += [f'step {s + 1}: ' + ['$x_t$', 'guess $\\hat x_0$'][k] for s in show_steps]
P.plot_designs(images, conds, titles)
plt.show()

A generative model gives us a *distribution* of designs, not just one. Here are several samples for the same condition, from different random noise:

In [ ]:
c_one = c_eval[:1].repeat(6, 1, 1, 1)
D_many = to_designs(sample(ddpm, c_one, seed=1))
C_many, _ = P.evaluate(D_many, [cond_eval[0]] * 6)
P.plot_designs(D_many, [cond_eval[0]] * 6, [f'C = {Cj:.3g}' for Cj in C_many])
plt.show()

## **Part C: Steering with Compliance Gradients**

The diffusion model imitates the dataset, good and bad designs alike. To push it toward *stiffer* brackets, we nudge its guess of the clean design downhill on compliance at every sampling step. For that, we need the gradient of compliance with respect to every pixel.

This is exactly what SIMP computes in every iteration. With the *adjoint method*, a single FEA solve gives the whole gradient: if element $e$ has density $\rho_e$ and stiffness proportional to $\rho_e$, then

$$\frac{\partial C}{\partial \rho_e} = -\, u_e^\top k_0\, u_e,$$

minus the strain energy that element would store at full stiffness. `P.compliance_gradient` returns $C$ and this gradient as an 80 × 80 image. Notice that it's **never positive**: adding material anywhere never makes the bracket softer.

In [ ]:
j = 0
C_j, dC = P.compliance_gradient(D_ddpm[j].astype(float), cond_eval[j])

fig, axs = plt.subplots(1, 2, figsize=(7.5, 3.6))
P.plot_design(D_ddpm[j], cond_eval[j], axs[0], title=f'DDPM design: C = {C_j:.3g}')
im = axs[1].imshow(np.log10(-dC + 1e-12), cmap='magma', vmin=np.log10(-dC + 1e-12).max() - 4)
axs[1].contour(D_ddpm[j], levels=[0.5], colors='white', linewidths=0.8)      # outline of the design
axs[1].set_title('log10(-dC/dρ): where material helps most', fontsize=9); axs[1].axis('off')
plt.colorbar(im, ax=axs[1], fraction=0.046); plt.tight_layout(); plt.show()

The gradient is largest in the empty space that gets stretched the most when the bracket deforms, typically right around the load and in gaps between members: filling those in would stiffen the bracket the most. It is small inside solid material, which barely strains, and in corners that don't move at all.

**Guidance.** At each sampling step, we take the model's guess $\hat x_0$, turn it into densities $\rho = (\hat x_0 + 1)/2$, and nudge it downhill on $\log C$:

$$\hat x_0 \leftarrow \hat x_0 - w \, \frac{\nabla \log C}{\text{std}(\nabla \log C)}$$

The DDIM step then continues from the nudged guess, so the network gets to clean up after each nudge. We stop nudging for the last 30% of the steps, so the final cleanup is the network's alone. Dividing by the gradient's standard deviation makes the weight $w$ roughly "pixel values moved per step". With $w = 0$ we get the plain DDPM back.

In [ ]:
def fea_guidance(x0, conds, weight):
    """Nudge each clean-design guess downhill on its true log-compliance: one FEA solve per design."""
    rho = ((x0[:, 0] + 1) / 2).cpu().numpy()                     # [-1, 1] -> densities in [0, 1]
    C, dC = P.compliance_gradients(rho, conds)                   # [n], [n x 80 x 80]
    grad = torch.tensor(dC / C[:, None, None], dtype=torch.float32, device=device).unsqueeze(1)   # gradient of log C
    grad = grad / grad.std(dim=(1, 2, 3), keepdim=True)
    return (x0 - weight * grad).clamp(-1, 1)

Let's generate designs for 30 test conditions with and without strong guidance, and check both with FEA. FEA runs on the CPU, so the guided run takes a minute or two (more on Colab).

In [ ]:
n_guide = 30
w = 3.0
D_plain = to_designs(sample(ddpm, c_eval[:n_guide]))           # same starting noise as the guided run below
C_plain, vf_plain = P.evaluate(D_plain, cond_eval[:n_guide])
t0 = time.time()
D_guided = to_designs(sample(ddpm, c_eval[:n_guide], guide=lambda x0: fea_guidance(x0, cond_eval[:n_guide], w)))
print(f'Guided generation took {time.time() - t0:.0f} s')
C_guided, vf_guided = P.evaluate(D_guided, cond_eval[:n_guide])

for name, C, vf in [('unguided', C_plain, vf_plain), (f'guided, w = {w}', C_guided, vf_guided)]:
    print(f'{name:15s}  median compliance / dataset: {np.median(C / C_data[:n_guide]):.2f}   '
          f'mean volume / cap: {vf.mean():.2f}   over the cap: {(vf > 1).mean():.0%}')

In [ ]:
P.plot_designs(np.concatenate([D_plain[:6], D_guided[:6]]), np.concatenate([cond_eval[:6]] * 2),
               [f'Unguided: C = {C_plain[j]:.3g}, vol/cap = {vf_plain[j]:.2f}' for j in range(6)]
               + [f'Guided: C = {C_guided[j]:.3g}, vol/cap = {vf_guided[j]:.2f}' for j in range(6)])
plt.show()

The guidance works: the guided designs are stiffer. But look at *how* they got there. The gradient is never positive, so the steepest way down is always to add material, and nothing in our guidance knows about the volume cap. The members grow thick, and the guided designs blow through their volume budget. In design terms, we optimized the objective and ignored the constraint. (Try a few other values of `w` to see the trade-off move.)

SIMP doesn't have this problem: every SIMP update enforces the volume constraint (the optimality-criteria step in `P.simp` searches for a Lagrange multiplier that keeps the volume at the cap). Our guidance has no such step. Yet.

## **Reflection Questions**

1. **Knowledge check:** In your run, how long did SIMP take per design, and how long did the diffusion model take? SIMP needs no training data at all. When would you rather use SIMP, and when a generative model?
2. **Fix the volume violations:** Modify the guidance so that designs stay under their volume cap while still improving compliance. Some ideas: make the nudge volume-neutral so it moves material instead of adding it (subtract the gradient's mean, like SIMP's Lagrange multiplier), add a volume penalty to the quantity the guidance descends, or keep only the most solid `vf_cap × 6400` pixels at the end. Compare compliance and volume with the unguided DDPM.

**Optional, but recommended for CP3:**

3. **More data:** What are some ways to expand the training dataset? Hint: think about transformations of existing designs (what has to happen to the condition when you mirror or rotate a bracket?) and about generating new designs. Try one and see whether the DDPM improves.
4. **A learned surrogate:** FEA guidance needs one CPU solve per design per step. Train a CNN that predicts $\log C$ from a design and its condition image (on the training designs), and guide with its gradient instead, using `torch.autograd`. How much faster is it? Do the guided designs get stiffer according to the *true* FEA? For strongly guided designs, compare the surrogate's predicted compliance with the FEA compliance, and explain what you see.
5. **Sampling steps:** Generate designs with 10 and with 200 DDIM steps instead of 50. How do generation time and design quality change?
6. **Seeding SIMP:** SIMP is a local optimizer, so where it ends up depends on where it starts. `P.simp(cond, x0=...)` starts it from your own densities instead of a uniform plate. Seed it with DDPM designs and compare the final compliance with unseeded SIMP: how often does the seeded run find a better, a worse, or about the same optimum? SIMP can't grow material where the seed is exactly 0, so blend the seed with the uniform start (e.g. `x0 = a * design + (1 - a) * vf_cap`) and try a few values of `a`. Is the generative model a useful warm start?